# Interview Preparation: Senior AWS DevOps Engineer

This notebook contains a curated set of interview questions, architectural scenarios, and technical deep-dives based on the **Smart Working Senior AWS DevOps Engineer** JD. Use this to prepare for high-level infrastructure roles focusing on AWS, EKS, Terraform, and Observability.

---

## 1. AWS Infrastructure & Networking

### Q1: Multi-Region VPC Connectivity
**Question:** Explain how you would design a multi-region VPC architecture for a global platform. How do you handle CIDR overlaps and ensure private connectivity (latency-sensitive) between services?

**Key Points to Cover:**
- VPC Peering vs. Transit Gateway.
- Shared Services VPC pattern.
- Route Tables and Security Group referencing across peered VPCs.
- Global Accelerator for entry points.

### Q2: EKS Networking & Security
**Question:** How do you implement the principle of least privilege within an EKS cluster? How do you ensure that Pod A cannot talk to Pod B unless explicitly permitted?

**Key Points to Cover:**
- Kubernetes Network Policies (Calico/Cilium).
- IAM Roles for Service Accounts (IRSA).
- Pod Security Admissions (PSA).
- App Mesh or Istio for mTLS.

## 2. Infrastructure as Code (Terraform)

### Q3: Terraform State & Collaboration
**Question:** In a small team where everyone has 'hands-on' ownership, how do you prevent state file corruption or manual 'drift' from breaking the pipeline?

**Key Points to Cover:**
- Remote State with S3 + DynamoDB locking.
- Terraform Cloud/Enterprise vs. DIY GitLab CI runners.
- Drift Detection (e.g., `terraform plan` on a schedule).
- Using `tflint` and `checkov` in CI.

### Q4: Refactoring Production Infra
**Question:** You need to rename a production RDS instance that was created with a generic name in Terraform. How do you do this without downtime or losing data?

**Key Points to Cover:**
- `terraform state mv` vs. `moved` blocks (TF 1.1+).
- Lifecycle rules (`prevent_destroy`).
- RDS snapshot strategies before state manipulation.

## 3. Kubernetes & Container Orchestration

### Q5: EKS Cluster Upgrades
**Question:** Describe your process for upgrading an EKS cluster from 1.28 to 1.30. What are the common 'gotchas' regarding API deprecations and Node Groups?

**Key Points to Cover:**
- Blue/Green Cluster upgrade vs. In-place node rolling.
- `pluto` or `kubent` for checking deprecated APIs.
- Upgrading Add-ons (CoreDNS, kube-proxy, VPC CNI).
- PDB (Pod Disruption Budgets) importance.

### Q6: Advanced Auto-scaling
**Question:** When would you choose **Karpenter** over the standard **Cluster Autoscaler**? How does Karpenter improve cost efficiency?

**Key Points to Cover:**
- Bin-packing vs. Group-based scaling.
- Spot instance integration and graceful termination.
- Faster spin-up times for nodes.

## 4. Observability & SRE

### Q7: The Golden Signals
**Question:** We use Grafana and Prometheus. If the 'Error Rate' signal spikes for a specific API Gateway, how do you correlate that with backend EKS logs in Splunk or ELK?

**Key Points to Cover:**
- Trace ID injection (OpenTelemetry).
- Log aggregation metadata (Namespace, PodID labels).
- CloudWatch Insights for quick filtering.

### Q8: Incident Response & Post-Mortems
**Question:** Describe a time you 'broke' production. What happened, how did you fix it, and what did your 'Blameless Post-Mortem' reveal?

**Key Points to Cover:**
- Transparency and communication during the outage.
- Technical root cause (MTTR vs. MTTD).
- Action items to prevent recurrence (Automation/Checks).

## 5. Scripting & Automation Challenges

### Scenario: The AMI Pipeline
**Task:** Write a pseudo-script (Python or Bash) that uses the AWS CLI to find the latest 'Golden AMI' and triggers a Terraform run if a newer one is found.

```python
import boto3
import subprocess

ec2 = boto3.client('ec2')

def get_latest_ami(filter_name):
    images = ec2.describe_images(
        Filters=[{'Name': 'name', 'Values': [filter_name]}],
        Owners=['self']
    )['Images']
    
    return sorted(images, key=lambda x: x['CreationDate'], reverse=True)[0]['ImageId']

# Logic to compare with current TF var and trigger GitLab CI Pipeline
```